# MLOps Assignment 2 — Goodreads Genre Classification

**Fine-tuning DistilBERT on UCSD Goodreads reviews — tracked with W&B, published to Hugging Face Hub.**

- **Author:** Rajath S M (g25ait2079) · PGD AI · IIT Jodhpur
- **Platform:** Kaggle Notebook (GPU T4 x2)
- **Tracking:** Weights & Biases
- **Model registry:** Hugging Face Hub

This notebook is the single executable deliverable. It loads UCSD Goodreads reviews for 8 genres, fine-tunes `distilbert-base-cased`, logs every metric to W&B, saves the classification report as a W&B Artifact, and finally pushes the trained model + tokenizer to a public HF repo.


## 0. Kaggle Setup (one-time per notebook)

**Before running anything below**, in the Kaggle UI:
1. **Settings → Accelerator → GPU T4 x2**
2. **Settings → Internet → On** (needed for dataset download + HF/W&B pushes)
3. **Add-ons → Secrets → Add:**
   - `WANDB_API_KEY` — get from https://wandb.ai/settings
   - `HF_TOKEN` — get from https://huggingface.co/settings/tokens (needs **Write** permission)
4. Check both secrets are **Attached to notebook**.


## 1. Load API tokens from Kaggle Secrets

In [ ]:
from kaggle_secrets import UserSecretsClient
import os

secrets = UserSecretsClient()
os.environ['WANDB_API_KEY'] = secrets.get_secret('WANDB_API_KEY')
os.environ['HF_TOKEN']      = secrets.get_secret('HF_TOKEN')

print("Secrets loaded.")


## 2. Install / upgrade libraries

In [ ]:
!pip install -q -U transformers wandb huggingface_hub


## 3. Imports & parameters

In [ ]:
from collections import defaultdict
import random, json, gzip, pickle, os
import numpy as np
import pandas as pd
import requests
import torch

from sklearn.metrics import accuracy_score, f1_score, classification_report
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments,
)
import wandb
from huggingface_hub import login as hf_login

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

model_name   = 'distilbert-base-cased'
device_name  = 'cuda' if torch.cuda.is_available() else 'cpu'
max_length   = 512
HF_REPO_ID   = 'Rajath-g25ait2079/distilbert-goodreads-genres'
WANDB_PROJECT = 'mlops-assignment2'
RUN_NAME     = 'distilbert-goodreads-run-1'

print('Device:', device_name)
print('Model :', model_name)


## 4. Load & sample Goodreads reviews per genre

In [ ]:
genre_url_dict = {
    'poetry':                 'https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_poetry.json.gz',
    'children':               'https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_children.json.gz',
    'comics_graphic':         'https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_comics_graphic.json.gz',
    'fantasy_paranormal':     'https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_fantasy_paranormal.json.gz',
    'history_biography':      'https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_history_biography.json.gz',
    'mystery_thriller_crime': 'https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_mystery_thriller_crime.json.gz',
    'romance':                'https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_romance.json.gz',
    'young_adult':            'https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_young_adult.json.gz',
}


def load_reviews(url, head=10000, sample_size=2000):
    """Stream a gzipped JSON of reviews and return a random sample of review_text strings."""
    reviews = []
    response = requests.get(url, stream=True, timeout=60)
    response.raise_for_status()
    with gzip.open(response.raw, 'rt', encoding='utf-8') as fh:
        for i, line in enumerate(fh):
            reviews.append(json.loads(line)['review_text'])
            if i + 1 >= head:
                break
    return random.sample(reviews, min(sample_size, len(reviews)))


genre_reviews_dict = {}
for genre, url in genre_url_dict.items():
    print(f'Loading {genre} ...')
    genre_reviews_dict[genre] = load_reviews(url, head=10000, sample_size=2000)
    print(f'  -> {len(genre_reviews_dict[genre])} reviews')

# Persist locally for fallback / re-runs
with open('genre_reviews_dict.pickle', 'wb') as fh:
    pickle.dump(genre_reviews_dict, fh)


### Fallback if automatic download fails

If any of the URLs above fail on Kaggle, download the `.json.gz` files manually from
<https://mengtingwan.github.io/data/goodreads.html#datasets>, upload them as a Kaggle **Dataset**,
attach it to this notebook, and replace `requests.get(url, stream=True)` with a local file open
on `/kaggle/input/<your-dataset>/...`.


## 5. Train / test split (800 train + 200 test per genre)

In [ ]:
train_texts, train_labels = [], []
test_texts,  test_labels  = [], []

for genre, reviews in genre_reviews_dict.items():
    sampled = random.sample(reviews, 1000)
    for r in sampled[:800]:
        train_texts.append(r); train_labels.append(genre)
    for r in sampled[800:]:
        test_texts.append(r);  test_labels.append(genre)

print('Train:', len(train_texts), '| Test:', len(test_texts))
print('Genres:', sorted(set(train_labels)))


## 6. Tokenize + build label maps + encode

In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained(model_name)

unique_labels = sorted(set(train_labels))
label2id = {label: i for i, label in enumerate(unique_labels)}
id2label = {i: label for label, i in label2id.items()}
NUM_LABELS = len(unique_labels)
print('Labels:', label2id)

train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=max_length)
test_encodings  = tokenizer(test_texts,  truncation=True, padding=True, max_length=max_length)

train_labels_encoded = [label2id[y] for y in train_labels]
test_labels_encoded  = [label2id[y] for y in test_labels]


## 7. Custom Torch dataset

In [ ]:
class GoodreadsDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)


train_dataset = GoodreadsDataset(train_encodings, train_labels_encoded)
test_dataset  = GoodreadsDataset(test_encodings,  test_labels_encoded)


## 8. Load the pre-trained DistilBERT model

In [ ]:
model = DistilBertForSequenceClassification.from_pretrained(
    model_name,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
).to(device_name)


## 9. Initialise W&B run + define compute_metrics

In [ ]:
wandb.login(key=os.environ['WANDB_API_KEY'])

wandb.init(
    project=WANDB_PROJECT,
    name=RUN_NAME,
    config={
        'model':         model_name,
        'epochs':        3,
        'batch_size':    16,
        'learning_rate': 3e-5,
        'max_length':    max_length,
        'dataset':       'UCSD Goodreads',
        'num_labels':    NUM_LABELS,
        'platform':      'Kaggle',
    },
)


def compute_metrics(pred):
    labels = pred.label_ids
    preds  = pred.predictions.argmax(-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1':       f1_score(labels, preds, average='weighted'),
    }


## 10. Training arguments + Trainer

In [ ]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=3e-5,
    warmup_steps=100,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    report_to='wandb',
    run_name=RUN_NAME,
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)


## 11. Train

In [ ]:
trainer.train()


## 12. Final evaluation + log final metrics to W&B

In [ ]:
eval_results = trainer.evaluate()
print(eval_results)

wandb.log({
    'final/loss':     eval_results['eval_loss'],
    'final/accuracy': eval_results['eval_accuracy'],
    'final/f1':       eval_results['eval_f1'],
})


## 13. Classification report → save as file → upload as W&B Artifact

In [ ]:
preds = trainer.predict(test_dataset).predictions.argmax(-1)
labels = [item['labels'].item() for item in test_dataset]

report = classification_report(
    labels, preds,
    target_names=[id2label[i] for i in range(NUM_LABELS)],
    output_dict=True,
)

with open('eval_report.json', 'w') as fh:
    json.dump(report, fh, indent=2)

# Also save a human-readable text version
with open('eval_report.txt', 'w') as fh:
    fh.write(classification_report(
        labels, preds,
        target_names=[id2label[i] for i in range(NUM_LABELS)],
    ))

artifact = wandb.Artifact('eval-report', type='evaluation')
artifact.add_file('eval_report.json')
artifact.add_file('eval_report.txt')
wandb.log_artifact(artifact)

print(open('eval_report.txt').read())


## 14. Push trained model + tokenizer to Hugging Face Hub

In [ ]:
hf_login(token=os.environ['HF_TOKEN'])

model.push_to_hub(HF_REPO_ID, private=False)
tokenizer.push_to_hub(HF_REPO_ID, private=False)

HF_URL = f'https://huggingface.co/{HF_REPO_ID}'
wandb.run.summary['huggingface_model'] = HF_URL
wandb.run.summary['final_accuracy']    = eval_results['eval_accuracy']
wandb.run.summary['final_f1']          = eval_results['eval_f1']
wandb.run.summary['final_loss']        = eval_results['eval_loss']

print('Pushed to:', HF_URL)


## 15. Close the W&B run

In [ ]:
wandb.finish()


## Done

After this notebook finishes:
- The W&B run page (under project `mlops-assignment2`) shows training curves + final metrics + the HF model URL.
- The HF repo `Rajath-g25ait2079/distilbert-goodreads-genres` is public and contains the trained weights and tokenizer.
- The classification report is attached to the W&B run as an Artifact named `eval-report`.

Take a screenshot of the W&B run charts for the report.
